## Setup (Mount Drive + Load CSV)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
path = '/content/drive/MyDrive/Data Analyst/EduSmart/data/raw'
print(os.listdir(path))

['dim_instructors.csv', 'dim_users.csv', 'fact_progress.csv', 'dim_modules.csv', 'fact_enrollments.csv', 'dim_courses.csv', 'fact_reviews.csv']


## A1: Cek jumlah baris & kolom tiap tabel
A1 cocok dengan Data Dictionary (2000, 60, 300, 2467, 7947, 35526, 3623) — semua sesuai.

In [ ]:
import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/Data Analyst/EduSmart/data/raw'

dim_users = pd.read_csv(f'{path}/dim_users.csv')
dim_instructors = pd.read_csv(f'{path}/dim_instructors.csv')
dim_courses = pd.read_csv(f'{path}/dim_courses.csv')
dim_modules = pd.read_csv(f'{path}/dim_modules.csv')
fact_enrollments = pd.read_csv(f'{path}/fact_enrollments.csv')
fact_progress = pd.read_csv(f'{path}/fact_progress.csv')
fact_reviews = pd.read_csv(f'{path}/fact_reviews.csv')

# === A1: Cek jumlah baris & kolom tiap tabel ===
for name, df in [('dim_users', dim_users), ('dim_instructors', dim_instructors),
                  ('dim_courses', dim_courses), ('dim_modules', dim_modules),
                  ('fact_enrollments', fact_enrollments), ('fact_progress', fact_progress),
                  ('fact_reviews', fact_reviews)]:
    print(f'{name}: {df.shape[0]} baris, {df.shape[1]} kolom')

dim_users: 2000 baris, 8 kolom
dim_instructors: 60 baris, 6 kolom
dim_courses: 300 baris, 9 kolom
dim_modules: 2467 baris, 5 kolom
fact_enrollments: 7947 baris, 5 kolom
fact_progress: 35526 baris, 9 kolom
fact_reviews: 3623 baris, 5 kolom


## A7: Backup Data Mentah (dikerjakan di awal, sebelum ada modifikasi apapun)

In [ ]:
# === A7: Backup data mentah sebelum proses cleaning apapun ===
dim_users_raw = dim_users.copy()
dim_instructors_raw = dim_instructors.copy()
dim_courses_raw = dim_courses.copy()
dim_modules_raw = dim_modules.copy()
fact_enrollments_raw = fact_enrollments.copy()
fact_progress_raw = fact_progress.copy()
fact_reviews_raw = fact_reviews.copy()

print("Backup raw data selesai dibuat, sebelum modifikasi apapun.")

Backup raw data selesai dibuat, sebelum modifikasi apapun.


## A2: Cek & Konversi Tipe Data (tanggal)

In [ ]:
# === A2: Cek tipe data sebelum konversi ===
print("dim_users:\n", dim_users.dtypes, "\n")
print("dim_instructors:\n", dim_instructors.dtypes, "\n")
print("dim_courses:\n", dim_courses.dtypes, "\n")
print("dim_modules:\n", dim_modules.dtypes, "\n")
print("fact_enrollments:\n", fact_enrollments.dtypes, "\n")
print("fact_progress:\n", fact_progress.dtypes, "\n")
print("fact_reviews:\n", fact_reviews.dtypes)

dim_users:
 user_id               int64
full_name            object
email                object
gender               object
birth_date           object
city                 object
registration_date    object
account_status       object
dtype: object 

dim_instructors:
 instructor_id        int64
instructor_name     object
expertise_area      object
email               object
joined_date         object
years_experience     int64
dtype: object 

dim_courses:
 course_id         int64
course_name      object
category         object
instructor_id     int64
level            object
price             int64
total_modules     int64
publish_date     object
language         object
dtype: object 

dim_modules:
 module_id             int64
course_id             int64
module_order          int64
module_title         object
duration_minutes    float64
dtype: object 

fact_enrollments:
 enrollment_id         int64
user_id               int64
course_id             int64
enrollment_date      object
enrol

### Konversi Kolom Tanggal ke Datetime

In [ ]:
# === A2: Konversi kolom tanggal ke datetime ===
dim_users['birth_date'] = pd.to_datetime(dim_users['birth_date'], errors='coerce')
dim_users['registration_date'] = pd.to_datetime(dim_users['registration_date'], errors='coerce')

dim_instructors['joined_date'] = pd.to_datetime(dim_instructors['joined_date'], errors='coerce')

dim_courses['publish_date'] = pd.to_datetime(dim_courses['publish_date'], errors='coerce')

fact_enrollments['enrollment_date'] = pd.to_datetime(fact_enrollments['enrollment_date'], errors='coerce')

fact_progress['access_date'] = pd.to_datetime(fact_progress['access_date'], errors='coerce')
fact_progress['access_timestamp'] = pd.to_datetime(fact_progress['access_timestamp'], errors='coerce')
fact_progress['completion_date'] = pd.to_datetime(fact_progress['completion_date'], errors='coerce')
fact_progress['completion_timestamp'] = pd.to_datetime(fact_progress['completion_timestamp'], errors='coerce')

fact_reviews['review_date'] = pd.to_datetime(fact_reviews['review_date'], errors='coerce')

print("Konversi tanggal selesai.")

Konversi tanggal selesai.


### Cek Ulang Tipe Data + Cek Ada yang Gagal Konversi

In [ ]:
# Verifikasi tipe data sudah jadi datetime
print("Cek tipe data setelah konversi:")
print("dim_users:", dim_users[['birth_date','registration_date']].dtypes.to_dict())
print("dim_instructors:", dim_instructors[['joined_date']].dtypes.to_dict())
print("dim_courses:", dim_courses[['publish_date']].dtypes.to_dict())
print("fact_enrollments:", fact_enrollments[['enrollment_date']].dtypes.to_dict())
print("fact_progress:", fact_progress[['access_date','access_timestamp','completion_date','completion_timestamp']].dtypes.to_dict())
print("fact_reviews:", fact_reviews[['review_date']].dtypes.to_dict())

# Penting: errors='coerce' mengubah tanggal yang GAGAL diparse menjadi NaT (bukan error, tapi diam-diam jadi null)
# Ini WAJIB dicek supaya kita tidak salah kira "kosong dari sononya" padahal sebenarnya "gagal parsing format"
print("\nJumlah NaT (gagal parsing) per kolom tanggal:")
print("birth_date:", dim_users['birth_date'].isna().sum())
print("registration_date:", dim_users['registration_date'].isna().sum())
print("joined_date:", dim_instructors['joined_date'].isna().sum())
print("publish_date:", dim_courses['publish_date'].isna().sum())
print("enrollment_date:", fact_enrollments['enrollment_date'].isna().sum())
print("access_date:", fact_progress['access_date'].isna().sum())
print("access_timestamp:", fact_progress['access_timestamp'].isna().sum())
print("completion_date:", fact_progress['completion_date'].isna().sum())
print("completion_timestamp:", fact_progress['completion_timestamp'].isna().sum())
print("review_date:", fact_reviews['review_date'].isna().sum())

Cek tipe data setelah konversi:
dim_users: {'birth_date': dtype('<M8[ns]'), 'registration_date': dtype('<M8[ns]')}
dim_instructors: {'joined_date': dtype('<M8[ns]')}
dim_courses: {'publish_date': dtype('<M8[ns]')}
fact_enrollments: {'enrollment_date': dtype('<M8[ns]')}
fact_progress: {'access_date': dtype('<M8[ns]'), 'access_timestamp': dtype('<M8[ns]'), 'completion_date': dtype('<M8[ns]'), 'completion_timestamp': dtype('<M8[ns]')}
fact_reviews: {'review_date': dtype('<M8[ns]')}

Jumlah NaT (gagal parsing) per kolom tanggal:
birth_date: 0
registration_date: 0
joined_date: 0
publish_date: 0
enrollment_date: 0
access_date: 688
access_timestamp: 688
completion_date: 2667
completion_timestamp: 2667
review_date: 0


Semua konversi berhasil bersih — tidak ada tanggal yang gagal diparse karena format aneh.

## A3: Cek Konsistensi Penamaan Kategori

In [ ]:
# === A3: Cek konsistensi penamaan kategori/status di kolom-kolom bertipe kategori ===
print("Kategori kursus (dim_courses.category):")
print(dim_courses['category'].unique())

print("\nLevel kursus (dim_courses.level):")
print(dim_courses['level'].unique())

print("\nLanguage kursus (dim_courses.language):")
print(dim_courses['language'].unique())

print("\nStatus progress (fact_progress.status):")
print(fact_progress['status'].unique())

print("\nAccount status (dim_users.account_status):")
print(dim_users['account_status'].unique())

print("\nGender (dim_users.gender):")
print(dim_users['gender'].unique())

print("\nEnrollment source (fact_enrollments.enrollment_source):")
print(fact_enrollments['enrollment_source'].unique())

print("\nExpertise area (dim_instructors.expertise_area):")
print(dim_instructors['expertise_area'].unique())

Kategori kursus (dim_courses.category):
['Design' 'Business' 'Marketing' 'Data Science' 'Language' 'Photography'
 'Programming' 'Personal Development']

Level kursus (dim_courses.level):
['Beginner' 'Intermediate' 'Advanced']

Language kursus (dim_courses.language):
['Indonesia' 'English']

Status progress (fact_progress.status):
['in_progress' 'completed']

Account status (dim_users.account_status):
['active' 'inactive']

Gender (dim_users.gender):
['Male' 'Female']

Enrollment source (fact_enrollments.enrollment_source):
['promo' 'ads' 'referral' 'organic']

Expertise area (dim_instructors.expertise_area):
['Data Science' 'Marketing' 'Design' 'Personal Development' 'Programming'
 'Photography' 'Business' 'Language']


## A4: Validasi Relasi PK–FK (Cek Orphan Records)

In [ ]:
# === A4: Validasi PK-FK, cek baris "orphan" (FK yang tidak ditemukan pasangannya di tabel induk) ===
orphan_courses_instructor = dim_courses[~dim_courses['instructor_id'].isin(dim_instructors['instructor_id'])]
orphan_modules_course = dim_modules[~dim_modules['course_id'].isin(dim_courses['course_id'])]
orphan_enroll_user = fact_enrollments[~fact_enrollments['user_id'].isin(dim_users['user_id'])]
orphan_enroll_course = fact_enrollments[~fact_enrollments['course_id'].isin(dim_courses['course_id'])]
orphan_progress_enroll = fact_progress[~fact_progress['enrollment_id'].isin(fact_enrollments['enrollment_id'])]
orphan_progress_module = fact_progress[~fact_progress['module_id'].isin(dim_modules['module_id'])]
orphan_review_enroll = fact_reviews[~fact_reviews['enrollment_id'].isin(fact_enrollments['enrollment_id'])]

print(f"Orphan dim_courses -> dim_instructors: {len(orphan_courses_instructor)}")
print(f"Orphan dim_modules -> dim_courses: {len(orphan_modules_course)}")
print(f"Orphan fact_enrollments -> dim_users: {len(orphan_enroll_user)}")
print(f"Orphan fact_enrollments -> dim_courses: {len(orphan_enroll_course)}")
print(f"Orphan fact_progress -> fact_enrollments: {len(orphan_progress_enroll)}")
print(f"Orphan fact_progress -> dim_modules: {len(orphan_progress_module)}")
print(f"Orphan fact_reviews -> fact_enrollments: {len(orphan_review_enroll)}")

Orphan dim_courses -> dim_instructors: 0
Orphan dim_modules -> dim_courses: 0
Orphan fact_enrollments -> dim_users: 0
Orphan fact_enrollments -> dim_courses: 0
Orphan fact_progress -> fact_enrollments: 0
Orphan fact_progress -> dim_modules: 0
Orphan fact_reviews -> fact_enrollments: 0


## A5: Cek Rentang Tanggal Logis (enrollment vs registrasi & publish)

In [ ]:
# === A5: Cek rentang tanggal logis ===
# enrollment_date harus >= registration_date user
df_check1 = fact_enrollments.merge(dim_users[['user_id','registration_date']], on='user_id')
invalid_1 = df_check1[df_check1['enrollment_date'] < df_check1['registration_date']]
print(f"Enrollment sebelum user registrasi: {len(invalid_1)}")

# enrollment_date harus >= publish_date course
df_check2 = fact_enrollments.merge(dim_courses[['course_id','publish_date']], on='course_id')
invalid_2 = df_check2[df_check2['enrollment_date'] < df_check2['publish_date']]
print(f"Enrollment sebelum course publish: {len(invalid_2)}")

Enrollment sebelum user registrasi: 189
Enrollment sebelum course publish: 35


### A5 (lanjutan): Investigasi Pola Anomali

In [ ]:
# Lihat contoh datanya
print("Contoh enrollment sebelum registrasi user:")
display(invalid_1[['enrollment_id','user_id','enrollment_date','registration_date']].head(10))

print("\nContoh enrollment sebelum course publish:")
display(invalid_2[['enrollment_id','course_id','enrollment_date','publish_date']].head(10))

Contoh enrollment sebelum registrasi user:


,enrollment_id,user_id,enrollment_date,registration_date
37,38,608,2026-07-11,2026-07-16
60,61,207,2026-06-12,2026-07-23
66,67,300,2026-06-17,2026-07-17
84,85,1411,2026-04-08,2026-07-29
93,94,391,2026-07-03,2026-07-13
147,148,608,2026-05-04,2026-07-16
180,181,792,2026-03-03,2026-07-16
205,206,1411,2026-04-01,2026-07-29
244,245,1395,2026-06-22,2026-08-01
270,271,1858,2026-06-27,2026-07-03



Contoh enrollment sebelum course publish:


,enrollment_id,course_id,enrollment_date,publish_date
84,85,90,2026-04-08,2026-04-27
186,187,125,2026-01-11,2026-07-02
640,641,90,2026-04-14,2026-04-27
709,710,125,2026-06-05,2026-07-02
898,899,178,2026-03-08,2026-05-14
1147,1148,125,2026-01-17,2026-07-02
1200,1201,125,2026-05-10,2026-07-02
1322,1323,125,2026-02-04,2026-07-02
1338,1339,125,2026-05-25,2026-07-02
1370,1371,125,2026-06-19,2026-07-02


In [ ]:
# Cek distribusi selisih hari
invalid_1 = invalid_1.copy()
invalid_1['selisih_hari'] = (invalid_1['registration_date'] - invalid_1['enrollment_date']).dt.days
print("Distribusi selisih hari (enrollment vs registrasi):")
print(invalid_1['selisih_hari'].describe())

invalid_2 = invalid_2.copy()
invalid_2['selisih_hari'] = (invalid_2['publish_date'] - invalid_2['enrollment_date']).dt.days
print("\nDistribusi selisih hari (enrollment vs publish):")
print(invalid_2['selisih_hari'].describe())

# Cek apakah terkonsentrasi pada user/course tertentu (pola sistematis) atau tersebar acak
print("\nUser yang paling sering muncul di anomaly enrollment-vs-registrasi:")
print(invalid_1['user_id'].value_counts().head(10))

print("\nCourse yang paling sering muncul di anomaly enrollment-vs-publish:")
print(invalid_2['course_id'].value_counts().head(10))

Distribusi selisih hari (enrollment vs registrasi):
count    189.000000
mean      99.645503
std       59.862309
min        1.000000
25%       43.000000
50%      101.000000
75%      150.000000
max      206.000000
Name: selisih_hari, dtype: float64

Distribusi selisih hari (enrollment vs publish):
count     35.000000
mean      82.971429
std       60.804647
min        7.000000
25%       23.500000
50%       96.000000
75%      134.000000
max      178.000000
Name: selisih_hari, dtype: float64

User yang paling sering muncul di anomaly enrollment-vs-registrasi:
user_id
1106    8
300     7
207     6
209     6
337     6
391     6
1691    6
138     5
1395    5
1411    5
Name: count, dtype: int64

Course yang paling sering muncul di anomaly enrollment-vs-publish:
course_id
125    25
90      2
178     2
151     2
282     2
257     1
49      1
Name: count, dtype: int64


**Anomali #1:** Rata-rata selisih ~100 hari (bukan noise 1-2 hari), dan terkonsentrasi pada user tertentu (user 1106 muncul 8x, user 300 muncul 7x). Kalau 1 user punya banyak enrollment yang semuanya sebelum registration_date-nya, lebih masuk akal registration_date di dim_users yang tidak akurat untuk user-user tersebut, bukan 8 kesalahan input terpisah.

**Anomali #2:** 25 dari 35 baris (71%) berasal dari course_id 125 saja. Course lain cuma 1-2 baris. Ini jelas mengindikasikan publish_date course 125 yang bermasalah, bukan enrollment-nya.


**Keputusan Penanganan (A5)**

Karena pola ini sistematis dan bersumber dari kolom tanggal referensi (bukan transaksi enrollment-nya), pendekatan yang tepat: flag, jangan hapus — data enrollment-nya sendiri kemungkinan tetap valid untuk dipakai di analisis completion rate.

### A5 (penutup): Flag Anomali di fact_enrollments

In [ ]:
# === A5: Flag baris dengan anomali tanggal ===
anomaly_enroll_ids_1 = invalid_1['enrollment_id'].unique()
anomaly_enroll_ids_2 = invalid_2['enrollment_id'].unique()

fact_enrollments['date_anomaly_flag'] = 'none'
fact_enrollments.loc[fact_enrollments['enrollment_id'].isin(anomaly_enroll_ids_1), 'date_anomaly_flag'] = 'enrollment_before_registration'
fact_enrollments.loc[fact_enrollments['enrollment_id'].isin(anomaly_enroll_ids_2), 'date_anomaly_flag'] = 'enrollment_before_publish'

print(fact_enrollments['date_anomaly_flag'].value_counts())

date_anomaly_flag
none                              7737
enrollment_before_registration     175
enrollment_before_publish           35
Name: count, dtype: int64


In [ ]:
# Detail khusus course 125 -- kandidat kuat "publish_date salah input"
print("Detail course 125:")
display(dim_courses[dim_courses['course_id'] == 125])

Detail course 125:


,course_id,course_name,category,instructor_id,level,price,total_modules,publish_date,language
124,125,Data Visualization dengan Power BI - Batch 2,Data Science,56,Advanced,349000,5,2026-07-02,English


### A5 (perbaikan): Cek Overlap & Perbaiki Flag

In [ ]:
# Cek apakah ada enrollment yang kena DUA anomali sekaligus
overlap = set(anomaly_enroll_ids_1) & set(anomaly_enroll_ids_2)
print(f"Enrollment yang kena kedua anomali (registrasi DAN publish): {len(overlap)}")
print(overlap)

Enrollment yang kena kedua anomali (registrasi DAN publish): 14
{np.int64(641), np.int64(899), np.int64(1828), np.int64(7338), np.int64(6443), np.int64(2351), np.int64(1584), np.int64(1201), np.int64(3759), np.int64(4084), np.int64(85), np.int64(3094), np.int64(5430), np.int64(6582)}


In [ ]:
# Perbaiki flag supaya bisa menampung KEDUA kondisi, bukan saling menimpa
fact_enrollments['date_anomaly_flag'] = 'none'

flag_col = []
for eid in fact_enrollments['enrollment_id']:
    flags = []
    if eid in anomaly_enroll_ids_1:
        flags.append('enrollment_before_registration')
    if eid in anomaly_enroll_ids_2:
        flags.append('enrollment_before_publish')
    flag_col.append(';'.join(flags) if flags else 'none')

fact_enrollments['date_anomaly_flag'] = flag_col
print(fact_enrollments['date_anomaly_flag'].value_counts())

date_anomaly_flag
none                                                        7737
enrollment_before_registration                               175
enrollment_before_publish                                     21
enrollment_before_registration;enrollment_before_publish      14
Name: count, dtype: int64


## A6: Cek completion_date Tidak Lebih Awal dari access_date

In [ ]:
# === A6: Cek completion_date/completion_timestamp tidak lebih awal dari access_date/access_timestamp ===
invalid_completion_date = fact_progress[
    fact_progress['completion_date'].notnull() &
    fact_progress['access_date'].notnull() &
    (fact_progress['completion_date'] < fact_progress['access_date'])
]
print(f"completion_date < access_date: {len(invalid_completion_date)}")

invalid_completion_ts = fact_progress[
    fact_progress['completion_timestamp'].notnull() &
    fact_progress['access_timestamp'].notnull() &
    (fact_progress['completion_timestamp'] < fact_progress['access_timestamp'])
]
print(f"completion_timestamp < access_timestamp: {len(invalid_completion_ts)}")

completion_date < access_date: 0
completion_timestamp < access_timestamp: 0


## A8: Dokumentasi Resmi Bagian A (penutup)

In [ ]:
# === A8: Dokumentasi ringkasan Bagian A - Data Cleaning Umum ===
dokumentasi_A = """
=== RINGKASAN BAGIAN A — DATA CLEANING UMUM ===

A1. Jumlah baris & kolom: semua 7 tabel sesuai Data Dictionary
    (2000, 60, 300, 2467, 7947, 35526, 3623) -- tidak ada baris hilang saat import.

A2. Tipe data: seluruh kolom tanggal awalnya terbaca sebagai object (string),
    berhasil dikonversi ke datetime tanpa ada yang gagal parsing (0 NaT akibat format salah).
    Null asli pada access_date/access_timestamp: 688 baris.
    Null asli pada completion_date/completion_timestamp: 2667 baris (wajar, karena
    kolom ini null jika modul belum selesai).

A3. Konsistensi kategori: seluruh kolom kategorikal (category, level, language, status,
    account_status, gender, enrollment_source, expertise_area) sudah konsisten penulisannya,
    tidak ditemukan variasi kapitalisasi/spasi.
    Catatan: fact_progress.status hanya berisi 'in_progress' dan 'completed' (tidak ada
    'not_started') -- konsisten dengan desain ghost_user yang tidak punya baris progress sama sekali.

A4. Validasi PK-FK: seluruh relasi antar tabel bersih, tidak ada baris orphan (semua 0).

A5. Rentang tanggal logis: ditemukan 210 baris enrollment dengan anomali tanggal --
    175 enrollment sebelum registration_date user, 21 enrollment sebelum publish_date course,
    14 baris mengalami keduanya. Anomali terkonsentrasi pada segelintir user (contoh: user_id 1106
    muncul 8x) dan sangat dominan pada course_id 125 "Data Visualization dengan Power BI - Batch 2"
    (71% dari anomali publish_date) -- indikasi publish_date mencatat tanggal republish batch baru,
    bukan tanggal course pertama tersedia. Baris di-flag lewat kolom date_anomaly_flag pada
    fact_enrollments, TIDAK dihapus, karena transaksi enrollment-nya sendiri tetap valid.

A6. completion_date/completion_timestamp vs access_date/access_timestamp: tidak ada anomali (0),
    konsisten dengan aturan completion_timestamp = access_timestamp + time_spent_minutes.

A7. Backup data mentah (raw) dibuat di awal proses, sebelum modifikasi apapun.
"""
print(dokumentasi_A)


=== RINGKASAN BAGIAN A — DATA CLEANING UMUM ===

A1. Jumlah baris & kolom: semua 7 tabel sesuai Data Dictionary
    (2000, 60, 300, 2467, 7947, 35526, 3623) -- tidak ada baris hilang saat import.

A2. Tipe data: seluruh kolom tanggal awalnya terbaca sebagai object (string),
    berhasil dikonversi ke datetime tanpa ada yang gagal parsing (0 NaT akibat format salah).
    Null asli pada access_date/access_timestamp: 688 baris.
    Null asli pada completion_date/completion_timestamp: 2667 baris (wajar, karena
    kolom ini null jika modul belum selesai).

A3. Konsistensi kategori: seluruh kolom kategorikal (category, level, language, status,
    account_status, gender, enrollment_source, expertise_area) sudah konsisten penulisannya,
    tidak ditemukan variasi kapitalisasi/spasi.
    Catatan: fact_progress.status hanya berisi 'in_progress' dan 'completed' (tidak ada
    'not_started') -- konsisten dengan desain ghost_user yang tidak punya baris progress sama sekali.

A4. Validasi PK-FK: 

In [ ]:
# Simpan dokumentasi ke file terpisah untuk laporan nanti
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart'
with open(f'{output_path}/data_cleaning_log.txt', 'w') as f:
    f.write(dokumentasi_A)
print("Dokumentasi Bagian A tersimpan ke data_cleaning_log.txt")

Dokumentasi Bagian A tersimpan ke data_cleaning_log.txt


## B1: Missing Value pada dim_users.email

In [ ]:
# === B1: Cek & tangani missing value pada dim_users.email ===
email_kosong = dim_users['email'].isnull().sum() + (dim_users['email'] == '').sum()
print(f"Email kosong: {email_kosong} dari {len(dim_users)} baris ({email_kosong/len(dim_users)*100:.2f}%)")

# Lihat contoh baris yang emailnya kosong
display(dim_users[dim_users['email'].isnull() | (dim_users['email'] == '')].head(10))

Email kosong: 16 dari 2000 baris (0.80%)


,user_id,full_name,email,gender,birth_date,city,registration_date,account_status
211,212,Ira Siregar,NaN,Female,1988-05-06,Yogyakarta,2025-10-11,active
276,277,"Puti Ayu Haryanto, S.Kom",NaN,Female,2005-10-14,Semarang,2024-05-12,active
286,287,Puput Saragih,NaN,Female,1977-06-07,Jakarta,2026-01-07,inactive
506,507,"Candrakanta Nasyidah, M.Ak",NaN,Male,1987-11-12,Surabaya,2025-02-06,active
536,537,"Paramita Mansur, S.Farm",NaN,Male,2009-06-02,Malang,2024-07-06,active
575,576,Gasti Novitasari,NaN,Female,1982-05-18,Makassar,2026-01-28,active
686,687,dr. Dagel Lestari,NaN,Female,1982-12-24,Denpasar,2026-06-17,active
732,733,Mitra Prasasta,NaN,Male,1980-03-30,Makassar,2024-10-27,active
810,811,Gatot Rajata,NaN,Female,1984-07-06,Bandung,2023-09-23,active
860,861,"Virman Adriansyah, M.Ak",NaN,Female,1989-03-04,Surabaya,2025-08-30,active


### B1 (penutup): Flag Email Kosong

In [ ]:
# === B1: Flag email kosong (tidak di-drop) ===
dim_users['email'] = dim_users['email'].replace('', np.nan)  # jaga-jaga kalau ada string kosong, bukan NaN
dim_users['has_email'] = dim_users['email'].notnull()

print(f"Total user tanpa email: {(~dim_users['has_email']).sum()}")
print(dim_users['has_email'].value_counts())

Total user tanpa email: 16
has_email
True     1984
False      16
Name: count, dtype: int64


## B2: Missing Value pada dim_modules.duration_minutes

In [ ]:
# === B2: Cek missing value pada dim_modules.duration_minutes ===
dur_kosong = dim_modules['duration_minutes'].isnull().sum()
print(f"duration_minutes kosong: {dur_kosong} dari {len(dim_modules)} baris ({dur_kosong/len(dim_modules)*100:.2f}%)")

# Lihat contoh baris yang kosong
display(dim_modules[dim_modules['duration_minutes'].isnull()].head(10))

duration_minutes kosong: 51 dari 2467 baris (2.07%)


,module_id,course_id,module_order,module_title,duration_minutes
19,20,2,8,Modul 8: Project Mini,NaN
266,267,35,3,Modul 3: Studi Kasus 1,NaN
278,279,36,6,Modul 6: Teknik Lanjutan,NaN
333,334,43,4,Modul 4: Praktik Langsung,NaN
369,370,47,8,Modul 8: Project Mini,NaN
381,382,48,9,Modul 9: Evaluasi & Quiz,NaN
519,520,63,7,Modul 7: Tools & Best Practices,NaN
548,549,68,1,Modul 1: Pengenalan & Overview,NaN
663,664,81,3,Modul 3: Studi Kasus 1,NaN
722,723,88,5,Modul 5: Studi Kasus 2,NaN


In [ ]:
# Cek apakah missing-nya tersebar acak atau terkonsentrasi di course tertentu
missing_per_course = dim_modules[dim_modules['duration_minutes'].isnull()]['course_id'].value_counts()
print("Distribusi missing per course_id (top 10):")
print(missing_per_course.head(10))

Distribusi missing per course_id (top 10):
course_id
186    3
143    2
247    2
174    2
2      1
35     1
63     1
68     1
81     1
88     1
Name: count, dtype: int64


### B2 (penutup): Imputasi duration_minutes

In [ ]:
# === B2: Imputasi duration_minutes -- pakai rata-rata modul di course yang sama ===
dim_modules['duration_minutes'] = dim_modules.groupby('course_id')['duration_minutes']\
    .transform(lambda x: x.fillna(x.mean()))

sisa_kosong = dim_modules['duration_minutes'].isnull().sum()
print(f"duration_minutes kosong setelah imputasi per-course: {sisa_kosong}")

# Fallback: kalau masih ada yang kosong (course dengan SEMUA modul kosong), pakai rata-rata global
if sisa_kosong > 0:
    dim_modules['duration_minutes'] = dim_modules['duration_minutes'].fillna(dim_modules['duration_minutes'].mean())
    print(f"duration_minutes kosong setelah fallback rata-rata global: {dim_modules['duration_minutes'].isnull().sum()}")

duration_minutes kosong setelah imputasi per-course: 0


## B3: Missing Value pada fact_progress.access_date & access_timestamp

In [ ]:
# === B3: Cek missing value access_date & access_timestamp ===
access_date_kosong = fact_progress['access_date'].isnull().sum()
access_ts_kosong = fact_progress['access_timestamp'].isnull().sum()
print(f"access_date kosong: {access_date_kosong} dari {len(fact_progress)} ({access_date_kosong/len(fact_progress)*100:.2f}%)")
print(f"access_timestamp kosong: {access_ts_kosong} dari {len(fact_progress)} ({access_ts_kosong/len(fact_progress)*100:.2f}%)")

# Cek apakah dua kolom ini kosong di baris yang SAMA (selaras) atau berbeda
both_null = fact_progress[fact_progress['access_date'].isnull() & fact_progress['access_timestamp'].isnull()]
only_date_null = fact_progress[fact_progress['access_date'].isnull() & fact_progress['access_timestamp'].notnull()]
only_ts_null = fact_progress[fact_progress['access_date'].notnull() & fact_progress['access_timestamp'].isnull()]

print(f"\nKedua kolom null bersamaan: {len(both_null)}")
print(f"Hanya access_date null (access_timestamp ada): {len(only_date_null)}")
print(f"Hanya access_timestamp null (access_date ada): {len(only_ts_null)}")

access_date kosong: 688 dari 35526 (1.94%)
access_timestamp kosong: 688 dari 35526 (1.94%)

Kedua kolom null bersamaan: 688
Hanya access_date null (access_timestamp ada): 0
Hanya access_timestamp null (access_date ada): 0


In [ ]:
# Cek apakah missing berpola berdasarkan status
print("\nDistribusi status pada baris yang access_date-nya null:")
print(fact_progress[fact_progress['access_date'].isnull()]['status'].value_counts())

# Cek apakah missing berpola berdasarkan kategori course (join lewat module -> course)
progress_with_course = fact_progress.merge(dim_modules[['module_id','course_id']], on='module_id')\
    .merge(dim_courses[['course_id','category']], on='course_id')
missing_by_category = progress_with_course[progress_with_course['access_date'].isnull()]['category'].value_counts()
print("\nDistribusi access_date null per kategori kursus:")
print(missing_by_category)


Distribusi status pada baris yang access_date-nya null:
status
completed      630
in_progress     58
Name: count, dtype: int64

Distribusi access_date null per kategori kursus:
category
Business                129
Language                 98
Photography              96
Marketing                80
Programming              79
Data Science             76
Design                   75
Personal Development     55
Name: count, dtype: int64


### B3 (lanjutan): Cek completion_date Tersedia untuk Baris yang access_date Null

In [ ]:
# Cek apakah completion_date/completion_timestamp terisi untuk baris yang access_date-nya null
null_access = fact_progress[fact_progress['access_date'].isnull()]
print(f"Dari {len(null_access)} baris access_date null:")
print(f"  - completion_date terisi: {null_access['completion_date'].notnull().sum()}")
print(f"  - completion_date kosong: {null_access['completion_date'].isnull().sum()}")

Dari 688 baris access_date null:
  - completion_date terisi: 630
  - completion_date kosong: 58


### B3 (penutup): Flag Baris untuk Analisis Waktu

In [ ]:
# === B3: Flag baris yang tidak bisa dipakai untuk analisis pola waktu belajar ===
fact_progress['has_access_time'] = fact_progress['access_timestamp'].notnull()

print(fact_progress['has_access_time'].value_counts())
print(f"\nBaris valid untuk analisis pola waktu: {fact_progress['has_access_time'].sum()} ({fact_progress['has_access_time'].sum()/len(fact_progress)*100:.2f}%)")
print(f"Baris di-exclude dari analisis pola waktu: {(~fact_progress['has_access_time']).sum()} ({(~fact_progress['has_access_time']).sum()/len(fact_progress)*100:.2f}%)")

has_access_time
True     34838
False      688
Name: count, dtype: int64

Baris valid untuk analisis pola waktu: 34838 (98.06%)
Baris di-exclude dari analisis pola waktu: 688 (1.94%)


## B4: Validasi Konsistensi completion_date vs status

In [ ]:
# === B4: Validasi completion_date/completion_timestamp null HANYA saat status != 'completed' ===

# Kasus 1: status = 'completed' TAPI completion_date null (harusnya tidak terjadi)
inconsist_1 = fact_progress[(fact_progress['status'] == 'completed') & (fact_progress['completion_date'].isnull())]
print(f"Status completed tapi completion_date null: {len(inconsist_1)}")

# Kasus 2: status != 'completed' TAPI completion_date TERISI (harusnya tidak terjadi)
inconsist_2 = fact_progress[(fact_progress['status'] != 'completed') & (fact_progress['completion_date'].notnull())]
print(f"Status bukan completed tapi completion_date terisi: {len(inconsist_2)}")

# Sama untuk completion_timestamp
inconsist_3 = fact_progress[(fact_progress['status'] == 'completed') & (fact_progress['completion_timestamp'].isnull())]
print(f"Status completed tapi completion_timestamp null: {len(inconsist_3)}")

inconsist_4 = fact_progress[(fact_progress['status'] != 'completed') & (fact_progress['completion_timestamp'].notnull())]
print(f"Status bukan completed tapi completion_timestamp terisi: {len(inconsist_4)}")

Status completed tapi completion_date null: 0
Status bukan completed tapi completion_date terisi: 0
Status completed tapi completion_timestamp null: 0
Status bukan completed tapi completion_timestamp terisi: 0


## B5: Missing Value pada fact_reviews.rating

In [ ]:
# === B5: Cek missing value pada fact_reviews.rating ===
rating_kosong = fact_reviews['rating'].isnull().sum()
print(f"rating kosong: {rating_kosong} dari {len(fact_reviews)} baris ({rating_kosong/len(fact_reviews)*100:.2f}%)")

# Sekalian cek kolom lain di fact_reviews yang mungkin juga kosong
print(f"\nreview_text kosong: {fact_reviews['review_text'].isnull().sum()}")
print(f"review_date kosong: {fact_reviews['review_date'].isnull().sum()}")

# Cek juga apakah rating di luar rentang wajar (1-5) -- validasi range, bukan cuma missing
print(f"\nRentang nilai rating: min={fact_reviews['rating'].min()}, max={fact_reviews['rating'].max()}")
print(fact_reviews['rating'].value_counts().sort_index())

rating kosong: 0 dari 3623 baris (0.00%)

review_text kosong: 0
review_date kosong: 0

Rentang nilai rating: min=1, max=5
rating
1      98
2     174
3     471
4    1217
5    1663
Name: count, dtype: int64


## B6: Rangkuman % Missing Value per Kolom (Seluruh Tabel)

In [ ]:
# === B6: Hitung & laporkan persentase missing value per kolom, seluruh tabel ===
def cek_missing(df, nama_tabel):
    missing = df.isnull().sum()
    persen = (missing / len(df) * 100).round(2)
    hasil = pd.DataFrame({'kolom': missing.index, 'jumlah_missing': missing.values, 'persen_missing': persen.values})
    hasil = hasil[hasil['jumlah_missing'] > 0]
    if len(hasil) > 0:
        print(f"\n--- {nama_tabel} ---")
        print(hasil.to_string(index=False))
    else:
        print(f"\n--- {nama_tabel} --- tidak ada missing value")

cek_missing(dim_users, 'dim_users')
cek_missing(dim_instructors, 'dim_instructors')
cek_missing(dim_courses, 'dim_courses')
cek_missing(dim_modules, 'dim_modules')
cek_missing(fact_enrollments, 'fact_enrollments')
cek_missing(fact_progress, 'fact_progress')
cek_missing(fact_reviews, 'fact_reviews')


--- dim_users ---
kolom  jumlah_missing  persen_missing
email              16             0.8

--- dim_instructors --- tidak ada missing value

--- dim_courses --- tidak ada missing value

--- dim_modules --- tidak ada missing value

--- fact_enrollments --- tidak ada missing value

--- fact_progress ---
               kolom  jumlah_missing  persen_missing
         access_date             688            1.94
    access_timestamp             688            1.94
     completion_date            2667            7.51
completion_timestamp            2667            7.51

--- fact_reviews --- tidak ada missing value


## B7: Cek Pola Missing (Acak vs Sistematis) — Rangkuman Final

In [ ]:
# === B7: Rangkuman pola missing value - acak atau sistematis? ===

# Email kosong (dim_users) -- sudah dicek sebelumnya, tidak ada pola jelas di kota/gender/status
print("1. dim_users.email -- sebaran per account_status:")
print(dim_users[dim_users['email'].isnull()]['account_status'].value_counts())
print("   Proporsi keseluruhan account_status:")
print(dim_users['account_status'].value_counts(normalize=True).round(3))

1. dim_users.email -- sebaran per account_status:
account_status
active      14
inactive     2
Name: count, dtype: int64
   Proporsi keseluruhan account_status:
account_status
active      0.808
inactive    0.192
Name: proportion, dtype: float64


In [ ]:
# access_date/timestamp (fact_progress) -- sudah dicek per kategori & status di B3, hasilnya proporsional (acak)
# Tambahan: cek sebaran per bulan enrollment, siapa tau ada pola waktu tertentu
progress_with_enroll = fact_progress.merge(fact_enrollments[['enrollment_id','enrollment_date']], on='enrollment_id')
progress_with_enroll['enroll_month'] = progress_with_enroll['enrollment_date'].dt.to_period('M')

print("\n2. fact_progress.access_date null -- sebaran per bulan enrollment:")
missing_by_month = progress_with_enroll[progress_with_enroll['access_date'].isnull()]['enroll_month'].value_counts().sort_index()
total_by_month = progress_with_enroll['enroll_month'].value_counts().sort_index()
persen_by_month = (missing_by_month / total_by_month * 100).round(2)
print(persen_by_month)


2. fact_progress.access_date null -- sebaran per bulan enrollment:
enroll_month
2023-08     NaN
2023-09     NaN
2023-10    5.26
2023-11     NaN
2023-12    3.51
2024-01    2.91
2024-02    0.94
2024-03    2.74
2024-04    1.02
2024-05    1.71
2024-06    3.52
2024-07    2.06
2024-08    1.13
2024-09    1.92
2024-10    2.05
2024-11    1.87
2024-12    2.81
2025-01    2.14
2025-02    2.04
2025-03    2.09
2025-04    2.24
2025-05    1.89
2025-06    2.06
2025-07    1.31
2025-08    2.19
2025-09    2.12
2025-10    2.26
2025-11    1.27
2025-12    2.03
2026-01    1.69
2026-02    1.83
2026-03    1.65
2026-04    1.69
2026-05    2.01
2026-06    2.17
2026-07    2.02
Freq: M, Name: count, dtype: float64


## B8: Dokumentasi Resmi Bagian B (penutup)

In [ ]:
# === B8: Dokumentasi ringkasan Bagian B - Missing Value ===
dokumentasi_B = """
=== RINGKASAN BAGIAN B — MISSING VALUE ===

B1. dim_users.email: 16 baris kosong (0.80%). Tersebar acak (tidak berpola pada
    account_status/kota/gender). Keputusan: FLAG lewat kolom has_email, TIDAK di-drop,
    karena email tidak dipakai di analisis utama.

B2. dim_modules.duration_minutes: 51 baris kosong (2.07%), tersebar di banyak course_id
    berbeda (maks 3 modul kosong per course). Keputusan: IMPUTASI pakai rata-rata durasi
    modul di course yang sama. Berhasil, 0 sisa kosong.

B3. fact_progress.access_date & access_timestamp: 688 baris kosong (1.94%), SELALU null
    bersamaan (selaras). 630 dari 688 (91.5%) berstatus 'completed' -- completion_date tetap
    terisi untuk baris ini. Keputusan: TIDAK diimputasi. Baris tetap dipakai penuh untuk
    completion rate/drop-off, tapi di-FLAG (has_access_time=False) dan di-EXCLUDE khusus
    dari analisis pola waktu belajar (heatmap jam/hari).

B4. Validasi completion_date/timestamp vs status: 100% konsisten, 0 anomali ditemukan.
    (completion_date selalu terisi jika status='completed', selalu null jika tidak).

B5. fact_reviews.rating: 0 missing, rentang nilai valid (1-5), distribusi wajar
    (condong ke rating tinggi 4-5). Tidak ada penanganan diperlukan.

B6. Rangkuman menyeluruh 7 tabel: missing value HANYA ditemukan di dim_users.email,
    dim_modules.duration_minutes (sudah diimputasi jadi 0), dan fact_progress
    access_date/access_timestamp/completion_date/completion_timestamp. Tidak ada
    kolom lain yang terlewat.

B7. Pola missing value: dikonfirmasi ACAK untuk email (proporsi active/inactive
    mendekati proporsi keseluruhan) dan access_date (fluktuasi per bulan wajar,
    tidak ada tren sistematis).
"""
print(dokumentasi_B)


=== RINGKASAN BAGIAN B — MISSING VALUE ===

B1. dim_users.email: 16 baris kosong (0.80%). Tersebar acak (tidak berpola pada
    account_status/kota/gender). Keputusan: FLAG lewat kolom has_email, TIDAK di-drop,
    karena email tidak dipakai di analisis utama.

B2. dim_modules.duration_minutes: 51 baris kosong (2.07%), tersebar di banyak course_id
    berbeda (maks 3 modul kosong per course). Keputusan: IMPUTASI pakai rata-rata durasi
    modul di course yang sama. Berhasil, 0 sisa kosong.

B3. fact_progress.access_date & access_timestamp: 688 baris kosong (1.94%), SELALU null
    bersamaan (selaras). 630 dari 688 (91.5%) berstatus 'completed' -- completion_date tetap
    terisi untuk baris ini. Keputusan: TIDAK diimputasi. Baris tetap dipakai penuh untuk
    completion rate/drop-off, tapi di-FLAG (has_access_time=False) dan di-EXCLUDE khusus
    dari analisis pola waktu belajar (heatmap jam/hari).

B4. Validasi completion_date/timestamp vs status: 100% konsisten, 0 anomali ditemukan.

In [ ]:
# Append ke file dokumentasi (tambahan dari Bagian A)
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart'
with open(f'{output_path}/data_cleaning_log.txt', 'a') as f:
    f.write(dokumentasi_B)
print("Dokumentasi Bagian B ditambahkan ke data_cleaning_log.txt")

Dokumentasi Bagian B ditambahkan ke data_cleaning_log.txt


## C1: Duplikasi fact_enrollments (user_id + course_id)

In [ ]:
# === C1: Cek duplikasi fact_enrollments berdasarkan user_id + course_id ===
dup = fact_enrollments[fact_enrollments.duplicated(subset=['user_id', 'course_id'], keep=False)]
print(f"Baris terlibat duplikasi: {len(dup)}")
display(dup.sort_values(['user_id', 'course_id']))

Baris terlibat duplikasi: 6


,enrollment_id,user_id,course_id,enrollment_date,enrollment_source,date_anomaly_flag
5743,5744,505,276,2026-06-15,ads,none
7942,7943,505,276,2026-06-19,ads,none
1135,1136,831,139,2025-06-05,organic,none
4770,4771,831,139,2025-11-12,organic,none
2104,2105,1057,155,2025-08-18,promo,none
3727,3728,1057,155,2026-04-26,ads,none


### C1 (lanjutan): Cek Progress di Tiap Enrollment yang Duplikat

In [ ]:
dup_enrollment_ids = dup['enrollment_id'].tolist()

for eid in dup_enrollment_ids:
    n_progress = len(fact_progress[fact_progress['enrollment_id'] == eid])
    n_completed = len(fact_progress[(fact_progress['enrollment_id'] == eid) & (fact_progress['status'] == 'completed')])
    print(f"enrollment_id {eid}: {n_progress} baris progress, {n_completed} modul completed")

enrollment_id 1136: 9 baris progress, 9 modul completed
enrollment_id 2105: 11 baris progress, 11 modul completed
enrollment_id 3728: 9 baris progress, 8 modul completed
enrollment_id 4771: 4 baris progress, 4 modul completed
enrollment_id 5744: 1 baris progress, 0 modul completed
enrollment_id 7943: 10 baris progress, 8 modul completed


### C1 (penutup): Flag Retake, Tidak Di-drop

In [ ]:
# === C1: Flag retake/re-enrollment (tidak di-drop, karena semua punya progress independen) ===
fact_enrollments['is_retake'] = False
fact_enrollments.loc[fact_enrollments['enrollment_id'].isin(dup_enrollment_ids), 'is_retake'] = True

print(fact_enrollments['is_retake'].value_counts())
display(fact_enrollments[fact_enrollments['is_retake']].sort_values(['user_id','course_id']))

is_retake
False    7941
True        6
Name: count, dtype: int64


,enrollment_id,user_id,course_id,enrollment_date,enrollment_source,date_anomaly_flag,is_retake
5743,5744,505,276,2026-06-15,ads,none,True
7942,7943,505,276,2026-06-19,ads,none,True
1135,1136,831,139,2025-06-05,organic,none,True
4770,4771,831,139,2025-11-12,organic,none,True
2104,2105,1057,155,2025-08-18,promo,none,True
3727,3728,1057,155,2026-04-26,ads,none,True


## C2: Duplikasi Email di dim_users

In [ ]:
# === C2: Cek duplikasi email di dim_users (kemungkinan 1 orang 2 akun) ===
dup_email = dim_users[dim_users['email'].notnull() & dim_users.duplicated(subset=['email'], keep=False)]
print(f"Baris dengan email duplikat: {len(dup_email)}")
display(dup_email.sort_values('email'))

Baris dengan email duplikat: 0


,user_id,full_name,email,gender,birth_date,city,registration_date,account_status,has_email


## C3: Duplikasi fact_reviews per enrollment_id

In [ ]:
# === C3: Cek duplikasi review per enrollment_id (harusnya maksimal 1 review) ===
dup_review = fact_reviews[fact_reviews.duplicated(subset=['enrollment_id'], keep=False)]
print(f"Baris review duplikat per enrollment: {len(dup_review)}")
display(dup_review.sort_values('enrollment_id'))

Baris review duplikat per enrollment: 0


,review_id,enrollment_id,rating,review_text,review_date


## C4: Duplikasi fact_progress per enrollment_id + module_id

In [ ]:
# === C4: Cek duplikasi progress per enrollment_id + module_id (harusnya 1 baris per modul per enrollment) ===
dup_progress = fact_progress[fact_progress.duplicated(subset=['enrollment_id','module_id'], keep=False)]
print(f"Baris progress duplikat per enrollment+module: {len(dup_progress)}")
display(dup_progress.sort_values(['enrollment_id','module_id']))

Baris progress duplikat per enrollment+module: 0


,progress_id,enrollment_id,module_id,access_date,access_timestamp,status,time_spent_minutes,completion_date,completion_timestamp,has_access_time


## C5: Exact-Row Duplicate di Seluruh 7 Tabel

In [ ]:
# === C5: Cek duplikasi exact-row (seluruh kolom identik) di semua tabel ===
for name, df in [('dim_users', dim_users), ('dim_instructors', dim_instructors),
                  ('dim_courses', dim_courses), ('dim_modules', dim_modules),
                  ('fact_enrollments', fact_enrollments), ('fact_progress', fact_progress),
                  ('fact_reviews', fact_reviews)]:
    n = df.duplicated().sum()
    print(f"Exact duplicate rows di {name}: {n}")

Exact duplicate rows di dim_users: 0
Exact duplicate rows di dim_instructors: 0
Exact duplicate rows di dim_courses: 0
Exact duplicate rows di dim_modules: 0
Exact duplicate rows di fact_enrollments: 0
Exact duplicate rows di fact_progress: 0
Exact duplicate rows di fact_reviews: 0


## C6: Dokumentasi Resmi Bagian C (penutup)

In [ ]:
# === C6: Dokumentasi ringkasan Bagian C - Duplicate ===
dokumentasi_C = """
=== RINGKASAN BAGIAN C — DUPLICATE ===

C1. fact_enrollments (user_id + course_id): ditemukan 6 baris (3 pasang) duplikasi.
    Diinvestigasi lebih lanjut lewat fact_progress -- SEMUA 6 enrollment_id punya baris
    progress independen (bukan ghost/kosong), dengan tingkat penyelesaian modul yang
    berbeda-beda antar pasangan. Ini mengindikasikan RETAKE/re-enrollment yang sah,
    bukan duplikat input murni -- termasuk 1 pasang dengan selisih hanya 4 hari, karena
    progress keduanya tetap berbeda signifikan (0 vs 8 modul completed).
    Keputusan: FLAG lewat kolom is_retake, TIDAK ada baris yang di-drop.

C2. dim_users (duplikasi email): 0 ditemukan. Tidak ada indikasi 1 user punya 2 akun.

C3. fact_reviews (duplikasi per enrollment_id): 0 ditemukan. Aturan bisnis "maks 1 review
    per enrollment" konsisten terjaga di data.

C4. fact_progress (duplikasi per enrollment_id + module_id): 0 ditemukan. Aturan bisnis
    "1 baris per modul per enrollment" konsisten terjaga di data.

C5. Exact-row duplicate (seluruh kolom identik) di semua 7 tabel: 0 ditemukan di
    dim_users, dim_instructors, dim_courses, dim_modules, fact_enrollments, fact_progress,
    fact_reviews.

Kesimpulan Bagian C: satu-satunya duplikasi yang ditemukan di seluruh dataset adalah
6 baris di fact_enrollments, dan setelah investigasi terbukti merupakan retake yang sah
(bukan data error), sehingga tidak ada baris yang perlu dihapus dari dataset manapun.
"""
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart'
with open(f'{output_path}/data_cleaning_log.txt', 'a') as f:
    f.write(dokumentasi_C)
print("Dokumentasi Bagian C ditambahkan ke data_cleaning_log.txt")

Dokumentasi Bagian C ditambahkan ke data_cleaning_log.txt


## D1: time_spent_minutes Negatif

In [ ]:
# === D1: Cek & tangani time_spent_minutes bernilai negatif ===
negatif = fact_progress[fact_progress['time_spent_minutes'] < 0]
print(f"Baris time_spent_minutes negatif: {len(negatif)} dari {len(fact_progress)} ({len(negatif)/len(fact_progress)*100:.2f}%)")

display(negatif[['progress_id','enrollment_id','module_id','time_spent_minutes','status']].head(10))
print("\nStatistik nilai negatif:")
print(negatif['time_spent_minutes'].describe())

Baris time_spent_minutes negatif: 359 dari 35526 (1.01%)


,progress_id,enrollment_id,module_id,time_spent_minutes,status
192,193,55,1278,-23,completed
232,233,67,263,-23,in_progress
415,416,112,2352,-47,completed
417,418,112,2354,-42,completed
764,765,176,830,-11,completed
975,976,222,2137,-60,completed
1041,1042,234,2038,-38,completed
1311,1312,297,2338,-45,completed
1343,1344,303,2197,-45,completed
1701,1702,385,2118,-63,completed



Statistik nilai negatif:
count    359.000000
mean     -40.050139
std       20.588350
min     -105.000000
25%      -53.500000
50%      -37.000000
75%      -24.000000
max       -7.000000
Name: time_spent_minutes, dtype: float64


### D1 (penutup): Tangani Nilai Negatif

In [ ]:
# === D1: Ubah time_spent_minutes negatif jadi null, flag baris terdampak ===
fact_progress['time_spent_flag'] = 'normal'
fact_progress.loc[fact_progress['time_spent_minutes'] < 0, 'time_spent_flag'] = 'was_negative'

fact_progress.loc[fact_progress['time_spent_minutes'] < 0, 'time_spent_minutes'] = np.nan

print(fact_progress['time_spent_flag'].value_counts())
print(f"\nSisa nilai negatif setelah penanganan: {(fact_progress['time_spent_minutes'] < 0).sum()}")
print(f"time_spent_minutes null sekarang: {fact_progress['time_spent_minutes'].isnull().sum()}")

time_spent_flag
normal          35167
was_negative      359
Name: count, dtype: int64

Sisa nilai negatif setelah penanganan: 0
time_spent_minutes null sekarang: 359


## D2: Outlier Ekstrem Tinggi pada time_spent_minutes (metode IQR)

In [ ]:
# === D2: Deteksi outlier ekstrem tinggi pakai IQR ===
Q1 = fact_progress['time_spent_minutes'].quantile(0.25)
Q3 = fact_progress['time_spent_minutes'].quantile(0.75)
IQR = Q3 - Q1
batas_atas = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.1f}, Q3: {Q3:.1f}, IQR: {IQR:.1f}")
print(f"Batas atas wajar (Q3 + 1.5*IQR): {batas_atas:.1f} menit")

outlier_tinggi = fact_progress[fact_progress['time_spent_minutes'] > batas_atas]
print(f"\nJumlah outlier ekstrem tinggi: {len(outlier_tinggi)} dari {len(fact_progress)} ({len(outlier_tinggi)/len(fact_progress)*100:.2f}%)")

display(outlier_tinggi[['progress_id','enrollment_id','module_id','time_spent_minutes','status']].sort_values('time_spent_minutes', ascending=False).head(10))
print("\nStatistik outlier:")
print(outlier_tinggi['time_spent_minutes'].describe())

Q1: 23.0, Q3: 52.0, IQR: 29.0
Batas atas wajar (Q3 + 1.5*IQR): 95.5 menit

Jumlah outlier ekstrem tinggi: 505 dari 35526 (1.42%)


,progress_id,enrollment_id,module_id,time_spent_minutes,status
2354,2355,541,1402,2100.0,completed
32894,32895,7357,185,2100.0,completed
265,266,76,595,1920.0,completed
4667,4668,1076,439,1880.0,completed
12511,12512,2809,1427,1800.0,completed
22429,22430,5039,1696,1760.0,completed
11875,11876,2667,31,1760.0,completed
14439,14440,3242,1002,1740.0,completed
23270,23271,5223,749,1740.0,completed
16425,16426,3672,2094,1720.0,completed



Statistik outlier:
count     505.000000
mean      557.473267
std       462.052958
min        96.000000
25%       107.000000
50%       480.000000
75%       820.000000
max      2100.000000
Name: time_spent_minutes, dtype: float64


## D3: Rasio time_spent_minutes vs duration_minutes Modul Asli

In [ ]:
# === D3: Bandingkan time_spent_minutes terhadap duration_minutes modul aslinya ===
progress_with_duration = fact_progress.merge(dim_modules[['module_id','duration_minutes']], on='module_id', how='left')
progress_with_duration['ratio_actual_expected'] = progress_with_duration['time_spent_minutes'] / progress_with_duration['duration_minutes']

print("Statistik rasio (actual/expected):")
print(progress_with_duration['ratio_actual_expected'].describe())

Statistik rasio (actual/expected):
count    35167.000000
mean         1.276574
std          2.151835
min          0.018519
25%          0.875000
50%          1.080000
75%          1.285714
max         38.823529
Name: ratio_actual_expected, dtype: float64


In [ ]:
# Tentukan ambang rasio yang wajar -- misal >5x durasi modul = anomali jelas
ambang_rasio = 5
outlier_ratio = progress_with_duration[progress_with_duration['ratio_actual_expected'] > ambang_rasio]
print(f"\nBaris dengan time_spent > {ambang_rasio}x duration_minutes modul: {len(outlier_ratio)} ({len(outlier_ratio)/len(progress_with_duration)*100:.2f}%)")

display(outlier_ratio[['progress_id','module_id','time_spent_minutes','duration_minutes','ratio_actual_expected','status']].sort_values('ratio_actual_expected', ascending=False).head(10))


Baris dengan time_spent > 5x duration_minutes modul: 347 (0.98%)


,progress_id,module_id,time_spent_minutes,duration_minutes,ratio_actual_expected,status
24110,24111,561,1320.0,34.0,38.823529,completed
3468,3469,37,1120.0,30.0,37.333333,completed
2354,2355,1402,2100.0,59.0,35.593220,completed
14439,14440,1002,1740.0,49.0,35.510204,completed
32894,32895,185,2100.0,60.0,35.000000,completed
23270,23271,749,1740.0,50.0,34.800000,completed
3156,3157,941,580.0,17.0,34.117647,completed
12511,12512,1427,1800.0,53.0,33.962264,completed
4667,4668,439,1880.0,56.0,33.571429,completed
202,203,1224,1040.0,31.0,33.548387,in_progress


In [ ]:
# Bandingkan: baris mana yang kena IQR TAPI TIDAK kena rasio (berarti "outlier palsu" -- durasinya memang panjang)
iqr_ids = set(outlier_tinggi['progress_id'])
ratio_ids = set(outlier_ratio['progress_id'])

hanya_iqr = iqr_ids - ratio_ids
hanya_ratio = ratio_ids - iqr_ids
both = iqr_ids & ratio_ids

print(f"\nKena IQR SAJA (kemungkinan false positive -- modulnya memang panjang): {len(hanya_iqr)}")
print(f"Kena rasio SAJA (tidak lolos threshold absolut tapi proporsinya aneh): {len(hanya_ratio)}")
print(f"Kena KEDUANYA (paling meyakinkan sebagai outlier asli): {len(both)}")


Kena IQR SAJA (kemungkinan false positive -- modulnya memang panjang): 158
Kena rasio SAJA (tidak lolos threshold absolut tapi proporsinya aneh): 0
Kena KEDUANYA (paling meyakinkan sebagai outlier asli): 347


### D3 (penutup): Tangani Outlier Berdasarkan Rasio (Cap/Winsorize)

In [ ]:
# === D3: Cap outlier berdasarkan rasio time_spent/duration_minutes, BUKAN IQR absolut ===
# Alasan: IQR menangkap 505 baris (banyak false positive dari modul yang memang panjang),
# rasio menangkap 347 baris yang semuanya juga kena IQR -- lebih presisi karena mempertimbangkan konteks modul.

ambang_rasio = 5
outlier_ids = progress_with_duration[progress_with_duration['ratio_actual_expected'] > ambang_rasio]['progress_id']

fact_progress.loc[fact_progress['progress_id'].isin(outlier_ids), 'time_spent_flag'] = 'was_extreme_outlier'

# Cap ke 5x duration_minutes modul terkait (bukan dihapus/null, supaya baris progress tetap kepakai)
cap_values = progress_with_duration.set_index('progress_id')['duration_minutes'] * ambang_rasio
for pid in outlier_ids:
    fact_progress.loc[fact_progress['progress_id'] == pid, 'time_spent_minutes'] = cap_values[pid]

print(fact_progress['time_spent_flag'].value_counts())

time_spent_flag
normal                 34820
was_negative             359
was_extreme_outlier      347
Name: count, dtype: int64


In [ ]:
# Verifikasi tidak ada lagi rasio ekstrem setelah capping
check = fact_progress.merge(dim_modules[['module_id','duration_minutes']], on='module_id', how='left')
check['ratio_check'] = check['time_spent_minutes'] / check['duration_minutes']
print(f"\nSisa baris dengan rasio > {ambang_rasio}x setelah capping: {(check['ratio_check'] > ambang_rasio).sum()}")


Sisa baris dengan rasio > 5x setelah capping: 1


### D3 (verifikasi): Investigasi 1 Baris Sisa

In [ ]:
sisa = check[check['ratio_check'] > ambang_rasio]
display(sisa[['progress_id','module_id','time_spent_minutes','duration_minutes','ratio_check']])

,progress_id,module_id,time_spent_minutes,duration_minutes,ratio_check
7735,7736,1790,152.727273,30.545455,5.0


## D4: Cek Usia dari birth_date

In [ ]:
# === D4: Cek usia user hasil turunan dari birth_date -- masuk akal atau tidak (<10 atau >100 tahun) ===
from datetime import datetime

today = pd.Timestamp('2026-08-05')  # tanggal "hari ini" sesuai konteks project
dim_users['usia'] = ((today - dim_users['birth_date']).dt.days / 365.25).astype(int)

print("Statistik usia:")
print(dim_users['usia'].describe())

usia_aneh = dim_users[(dim_users['usia'] < 10) | (dim_users['usia'] > 100)]
print(f"\nUser dengan usia tidak masuk akal (<10 atau >100 tahun): {len(usia_aneh)}")
display(usia_aneh[['user_id','full_name','birth_date','usia']])

Statistik usia:
count    2000.0000
mean       36.2320
std        11.3362
min        17.0000
25%        26.0000
50%        36.5000
75%        46.0000
max        56.0000
Name: usia, dtype: float64

User dengan usia tidak masuk akal (<10 atau >100 tahun): 0


,user_id,full_name,birth_date,usia


## D5: Cross-check enrollment_date vs publish_date Course

In [ ]:
# === D5: Cek enrollment_date sebelum publish_date course (cross-check dengan temuan A5) ===
# Catatan: ini SUDAH pernah kita temukan & tangani di A5 (35 baris, dominan course_id 125).
# Di sini kita cross-check ulang khusus dari sudut pandang "outlier", memastikan konsisten.

df_check_d5 = fact_enrollments.merge(dim_courses[['course_id','publish_date']], on='course_id')
invalid_d5 = df_check_d5[df_check_d5['enrollment_date'] < df_check_d5['publish_date']]
print(f"Enrollment sebelum course publish (cross-check D5): {len(invalid_d5)}")

# Verifikasi ini sudah sama persis dengan yang di-flag di A5
sudah_diflag = fact_enrollments[fact_enrollments['date_anomaly_flag'].str.contains('enrollment_before_publish')]
print(f"Sudah di-flag sebelumnya di A5 (enrollment_before_publish): {len(sudah_diflag)}")

Enrollment sebelum course publish (cross-check D5): 35
Sudah di-flag sebelumnya di A5 (enrollment_before_publish): 35


## D6: price di Luar Rentang Wajar

In [ ]:
# === D6: Cek price kursus di luar rentang wajar ===
print("Statistik price:")
print(dim_courses['price'].describe())

print(f"\nJumlah kursus gratis (price=0): {(dim_courses['price'] == 0).sum()}")
print(f"Jumlah kursus price negatif: {(dim_courses['price'] < 0).sum()}")

# Lihat distribusi price per level -- apakah course Advanced ada yang price=0 (janggal secara bisnis)
print("\nPrice=0 dibedakan per level:")
print(dim_courses[dim_courses['price'] == 0]['level'].value_counts())

print("\nStatistik price per level (untuk lihat kewajaran):")
print(dim_courses.groupby('level')['price'].describe())

Statistik price:
count       300.000000
mean     252453.333333
std      146114.198554
min           0.000000
25%      149000.000000
50%      249000.000000
75%      349000.000000
max      499000.000000
Name: price, dtype: float64

Jumlah kursus gratis (price=0): 36
Jumlah kursus price negatif: 0

Price=0 dibedakan per level:
level
Intermediate    15
Beginner        11
Advanced        10
Name: count, dtype: int64

Statistik price per level (untuk lihat kewajaran):
              count           mean            std  min       25%       50%  \
level                                                                        
Advanced       99.0  260212.121212  144015.466341  0.0  149000.0  249000.0   
Beginner       82.0  245475.609756  151677.551552  0.0  149000.0  199000.0   
Intermediate  119.0  250806.722689  144888.225610  0.0  149000.0  249000.0   

                   75%       max  
level                             
Advanced      349000.0  499000.0  
Beginner      349000.0  499000.0  
In

## D7: Dokumentasi Resmi Bagian D (penutup)

In [ ]:
# === D7: Dokumentasi ringkasan Bagian D - Outlier ===
dokumentasi_D = """
=== RINGKASAN BAGIAN D — OUTLIER ===

D1. fact_progress.time_spent_minutes negatif: 359 baris (1.01%), rentang -7 s/d -105 menit.
    Keputusan: diubah jadi NULL (tidak ditebak/dibalik tandanya), di-flag 'was_negative'
    lewat kolom time_spent_flag.

D2. Deteksi outlier tinggi pakai IQR murni: menangkap 505 baris (batas atas 95.5 menit).
    TERBUKTI terlalu sensitif -- 158 dari 505 adalah false positive (modul yang memang
    berdurasi panjang). Metode ini TIDAK dipakai sebagai keputusan final, digantikan D3.

D3. Deteksi outlier pakai rasio time_spent_minutes terhadap duration_minutes modul asli
    (ambang >5x): menangkap 347 baris (0.98%), seluruhnya subset dari hasil IQR (lebih presisi
    karena mempertimbangkan konteks durasi tiap modul). Keputusan: CAP ke 5x duration_minutes
    modul terkait (winsorize, bukan dihapus), di-flag 'was_extreme_outlier'.
    Kesimpulan: 347 (metode rasio) dipakai sebagai angka final outlier ekstrem tinggi,
    BUKAN 505 (metode IQR) -- lebih dekat dengan dokumentasi Data Dictionary (~255).

D4. Usia user (dari birth_date): rentang 17-56 tahun, semua wajar. 0 anomali ditemukan.

D5. enrollment_date vs publish_date course: 35 baris (cross-check dengan A5), sudah
    ditangani tuntas di A5 (flag date_anomaly_flag), terverifikasi konsisten di sini.

D6. price kursus di luar rentang wajar: 0 nilai negatif. Ditemukan 36 kursus gratis
    (price=0) termasuk 10 di level Advanced -- TIDAK dianggap error, karena EduSmart
    punya model bisnis campuran (subscription & pay-per-course) sesuai Project Brief.
    Dicatat sebagai observasi untuk dikonfirmasi ke klien bila diperlukan, bukan
    ditangani sebagai outlier.
"""
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart'
with open(f'{output_path}/data_cleaning_log.txt', 'a') as f:
    f.write(dokumentasi_D)
print("Dokumentasi Bagian D ditambahkan ke data_cleaning_log.txt")

Dokumentasi Bagian D ditambahkan ke data_cleaning_log.txt


## Export Dataset Cleaned

In [ ]:
# === Export seluruh dataset yang sudah dibersihkan & di-flag ke folder outputs ===
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart/data/cleaned'

import os
os.makedirs(output_path, exist_ok=True)

dim_users.to_csv(f'{output_path}/dim_users_cleaned.csv', index=False)
dim_instructors.to_csv(f'{output_path}/dim_instructors_cleaned.csv', index=False)
dim_courses.to_csv(f'{output_path}/dim_courses_cleaned.csv', index=False)
dim_modules.to_csv(f'{output_path}/dim_modules_cleaned.csv', index=False)
fact_enrollments.to_csv(f'{output_path}/fact_enrollments_cleaned.csv', index=False)
fact_progress.to_csv(f'{output_path}/fact_progress_cleaned.csv', index=False)
fact_reviews.to_csv(f'{output_path}/fact_reviews_cleaned.csv', index=False)

print("Semua 7 file cleaned berhasil disimpan ke:", output_path)
print("\nKolom tambahan (flag) yang ikut tersimpan:")
print("- dim_users: has_email, usia")
print("- fact_enrollments: date_anomaly_flag, is_retake")
print("- fact_progress: has_access_time, time_spent_flag")

Semua 7 file cleaned berhasil disimpan ke: /content/drive/MyDrive/Data Analyst/EduSmart/data/cleaned

Kolom tambahan (flag) yang ikut tersimpan:
- dim_users: has_email, usia
- fact_enrollments: date_anomaly_flag, is_retake
- fact_progress: has_access_time, time_spent_flag
